In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/input/title-header/ChatGPT Image Apr 16 2025 09_09_41 PM.png"))

## 🎓 PathFinder AI – Smart College Counseling Assistant
Helping high schoolers discover the right colleges, scholarships, and activities with Gen AI. AI-Powered College & Scholarship Advisor for First-Gen Students

### 🧠 Project Overview
💡 “What if every first-generation student had a personalized college counselor powered by AI?”

This project uses Generative AI + Retrieval-Augmented Generation (RAG) to provide personalized college and scholarship recommendations for students based on their interests, goals, and background — especially targeting first-generation college applicants.

### 🚀 What This App Does
🎓 Recommends best-fit colleges using your interests + profile

💰 Suggests relevant scholarships with eligibility info

🧠 Suggests extracurricular activities aligned with your academic and personal goals

🧭 Provides rationale for each suggestion, grounded in real data

### 🔧 Under the Hood
This project integrates:

Google Gemini for smart, structured recommendations

FAISS + Sentence Transformers for semantic search of a custom dataset

Retrieval-Augmented Generation (RAG) for grounded, accurate GenAI output

ipywidgets UI to interact with your profile and receive personalized guidance

### GenAI Capabilities Demonstrated

✅ Structured output (JSON mode)

✅ Grounding via FAISS-based retrieval

✅ Prompt engineering & format control

✅ Embeddings + vector search

✅ Context caching + similarity transparency

### 🧪 Try It Yourself!
👇 Scroll down to the “Interactive UI” section, type in your interests (e.g.,
“I'm a first-gen student interested in psychology and financial wellness”),
and click Get Recommendations to experience it in action.



In [40]:
!pip install faiss-cpu
!pip install feedparser

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [50]:
# %% [markdown]
# # 🎓 College & Scholarship Advisor (RAG + Gemini)
# This project combines Retrieval-Augmented Generation (RAG), Gemini, FAISS, and embeddings to recommend colleges and scholarships based on a student’s profile.
# 
# ✅ Powered by:
# - Google Gemini (via `google.generativeai`)
# - SentenceTransformers + FAISS
# - Kaggle UI with `ipywidgets`

# %% [code]
# 📥 Imports & Setup
import pandas as pd
import numpy as np
import json
import faiss
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

# %% [code]
# 🔐 Load Gemini API Key and College Score Card API Key
GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
COLLEGE_SCORECARD_API_KEY = UserSecretsClient().get_secret("COLLEGE_SCORECARD_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.0-flash")
chat = gemini_model.start_chat()

# %% [code]
# 📊 Load Datasets
# 🎓 Fetch live College Scorecard data
def fetch_live_college_data(limit=300, cache_path="college_scorecard_cache.json", use_cache=True):
    BASE_URL = "https://api.data.gov/ed/collegescorecard/v1/schools.json"

    if use_cache and os.path.exists(cache_path):
        print("🔁 Loading cached college data...")
        with open(cache_path, "r") as f:
            results = json.load(f)
    else:
        print("🔍 Fetching live college data...")
        params = {
            "api_key": COLLEGE_SCORECARD_API_KEY,
            "fields": "school.name,school.city,school.state,school.control,2022.cost.tuition.in_state,2022.admissions.admission_rate.overall,2022.admissions.sat_scores.average.overall",
            "per_page": 100
        }
        results = []
        pages = limit // 100
        for i in range(pages):
            params["page"] = i
            r = requests.get(BASE_URL, params=params)
            if r.status_code != 200:
                break
            results.extend(r.json()["results"])
            time.sleep(0.3)
        with open(cache_path, "w") as f:
            json.dump(results, f)

    df = pd.DataFrame(results).dropna()

    # Safely rename if exists
    df.rename(columns={
        "school.name": "College Name",
        "school.city": "City",
        "school.state": "State",
        "2022.cost.tuition.in_state": "Tuition",
        "2022.admissions.admission_rate.overall": "Acceptance Rate",
        "2022.admissions.sat_scores.average.overall": "Avg SAT"
    }, inplace=True)

    # Use fallback column names if 'Control' doesn't exist
    def get_value(row, primary, fallback):
        return row.get(primary) or row.get(fallback)

    # Handle description even if rename failed
    df["Description"] = df.apply(lambda row: (
        f"{get_value(row, 'College Name', 'school.name')} in "
        f"{get_value(row, 'City', 'school.city')}, "
        f"{get_value(row, 'State', 'school.state')} is a "
        f"Tuition: ${row.get('Tuition', row.get('2022.cost.tuition.in_state'))}, "
        f"Acceptance Rate: {row.get('Acceptance Rate', row.get('2022.admissions.admission_rate.overall')):.2f}, "
        f"Avg SAT: {row.get('Avg SAT', row.get('2022.admissions.sat_scores.average.overall'))}."
    ), axis=1)

    return df

# 🧠 Build FAISS index from descriptions
def build_faiss_index_from_descriptions(df, column="Description"):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(df[column].tolist())
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))
    return model, index, embeddings

# ✅ Run everything
college_df = fetch_live_college_data(limit=300)
scholarship_df = pd.read_csv('/kaggle/input/world-scholarships/Universities_Schoolarships_All_Around_the_World.csv', index_col=0)

# Drop rows with NaN in any of the relevant columns
required_cols = ["title", "degrees", "funds", "date", "location"]
filtered_df = scholarship_df.dropna(subset=required_cols).copy()

#Filter to only U.S.-based scholarships
scholarship_df = filtered_df[filtered_df["location"].str.lower() == "united-states"].copy()

# Rename the 'title' column to 'Scholarship Name' for consistency
scholarship_df.rename(columns={"title": "Scholarship Name"}, inplace=True)

# Build a formatted Description column
scholarship_df["Description"] = scholarship_df.apply(lambda row: (
    f"{row['Scholarship Name']} is a {row['degrees']} scholarship available in the United States.\n"
    f"Funding: {row['funds']}, Deadline: {row['date']}."
), axis=1)

# Optional: Preview or export
scholarship_df[["Scholarship Name", "Description"]].head()

embed_model, college_index, college_embeddings = build_faiss_index_from_descriptions(college_df)
_, scholarship_index, scholarship_embeddings = build_faiss_index_from_descriptions(scholarship_df)

print(f"✅ Loaded {len(college_df)} colleges and {len(scholarship_df)} scholarships.")

# %% [markdown]
# ## 🧠 CollegeRAGAdvisor Class (RAG Loop + Cache + FAISS Distances)

# %% [code]
class CollegeRAGAdvisor:
    def __init__(self, college_df, scholarship_df, chat, embedding_model, k_colleges=3, k_scholarships=2):
        self.college_df = college_df
        self.scholarship_df = scholarship_df
        self.chat = chat
        self.model = embedding_model
        self.k_colleges = k_colleges
        self.k_scholarships = k_scholarships
        self.cache = {}
        self._create_faiss_indices()

    def _create_faiss_indices(self):
        # Embed and index college descriptions
        self.college_embeddings = self.model.encode(self.college_df["Description"].tolist())
        self.college_index = faiss.IndexFlatL2(self.college_embeddings.shape[1])
        self.college_index.add(np.array(self.college_embeddings))

        # Embed and index scholarship descriptions
        self.scholarship_embeddings = self.model.encode(self.scholarship_df["Description"].tolist())
        self.scholarship_index = faiss.IndexFlatL2(self.scholarship_embeddings.shape[1])
        self.scholarship_index.add(np.array(self.scholarship_embeddings))

    def _retrieve_context(self, query):
        query_embedding = self.model.encode([query])

        college_dists, college_ids = self.college_index.search(np.array(query_embedding), self.k_colleges)
        top_colleges = self.college_df.iloc[college_ids[0]].copy()

        scholarship_dists, scholarship_ids = self.scholarship_index.search(np.array(query_embedding), self.k_scholarships)
        top_scholarships = self.scholarship_df.iloc[scholarship_ids[0]].copy()

        return top_colleges, top_scholarships, college_dists[0], scholarship_dists[0]

    def _build_prompt(self, query, colleges, scholarships, college_dists, scholarship_dists):
        context = "### College Info (with similarity scores):\n"
        for (idx, row), dist in zip(colleges.iterrows(), college_dists):
            context += f"- {row['College Name']} (Distance: {dist:.4f}): {row['Description']}\n"

        context += "\n### Scholarship Info (with similarity scores):\n"
        for (idx, row), dist in zip(scholarships.iterrows(), scholarship_dists):
            context += f"- {row['Scholarship Name']} (Distance: {dist:.4f}): {row['Description']}\n"

        full_prompt = (
            f"You are an expert college advisor. Use the context below to generate "
            f"personalized college and scholarship recommendations for the following user:\n\n"
            f"{context}\n"
            f"\n### User Profile:\n{query}\n"
            f"\nPlease return your answer in this JSON format:\n"
            f"```\n"
            f"{{\n"
            f'  "college_recommendations": [{{"college": "...", "rationale": "..."}}],\n'
            f'  "extracurricular_suggestions": [{{"activity": "...", "rationale": "..."}}],\n'
            f'  "scholarship_recommendation": {{"scholarship": "...", "eligibility": "..."}}\n'
            f"}}\n"
            f"```"
        )
        return full_prompt

    def recommend(self, user_query):
        if user_query in self.cache:
            print("🔁 Using cached result.")
            response_dict = self.cache[user_query]
        else:
            # ❗Reset Gemini chat context to avoid hallucinating from past queries
            self.chat = self.chat.model.start_chat()
            
            colleges, scholarships, college_dists, scholarship_dists = self._retrieve_context(user_query)
            prompt = self._build_prompt(user_query, colleges, scholarships, college_dists, scholarship_dists)
            response = self.chat.send_message(prompt)

            cleaned_text = response.text.strip()
            if cleaned_text.startswith("```json"):
                cleaned_text = cleaned_text[7:]
            if cleaned_text.endswith("```"):
                cleaned_text = cleaned_text[:-3]

            try:
                response_dict = json.loads(cleaned_text)
                self.cache[user_query] = response_dict
            except json.JSONDecodeError:
                print("❌ Failed to parse JSON. Raw output:\n", cleaned_text)
                return

        # Display as Markdown (optional rendering)
        from IPython.display import display, Markdown
        md = "# 🎓 College Recommendations\n"
        for college in response_dict["college_recommendations"]:
            md += f"## {college['college']}\n{college['rationale']}\n\n"

        md += "## 🏫 Extracurricular Suggestions\n"
        for item in response_dict["extracurricular_suggestions"]:
            md += f"- **{item['activity']}**\n  - {item['rationale']}\n"

        sch = response_dict["scholarship_recommendation"]
        md += "\n## 💰 Scholarship Recommendation\n"
        md += f"**Scholarship:** {sch['scholarship']}\n\n"
        md += f"**Eligibility:** {sch['eligibility']}\n"

        display(Markdown(md))

# %% [code]
advisor = CollegeRAGAdvisor(
    college_df=college_df,
    scholarship_df=scholarship_df,
    chat=chat,
    embedding_model=embed_model,
    k_colleges=5,
    k_scholarships=3
)

# %% [markdown]
# ## 🖼️ Interactive UI with `ipywidgets`

# %% [code]
def run_ui(advisor):
    query_input = widgets.Textarea(
        value='',  # Empty on load
        placeholder=(
            "Example: I'm a first-gen student interested in computer science and need financial aid.\n"
            "I'm part of the robotics club, play soccer, and volunteer at the library."
        ),
        description='Your Profile:',
        layout=widgets.Layout(width='100%', height='120px')
    )

    run_button = widgets.Button(
        description='Get Recommendations',
        button_style='primary',
        tooltip='Run RAG + Gemini',
        icon='💡'
    )

    output = widgets.Output()

    example_box = widgets.HTML(
        "<b>💡 Tip:</b> Mention your goals (major, degree), need for scholarships, and extracurriculars.<br>"
        "Optional: share certifications, sports, or personal story elements."
    )

    def on_click(b):
        with output:
            clear_output()
            advisor.recommend(query_input.value)

    run_button.on_click(on_click)

    display(example_box, query_input, run_button, output)

# Launch the UI
run_ui(advisor)


🔁 Loading cached college data...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Loaded 42 colleges and 107 scholarships.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

HTML(value='<b>💡 Tip:</b> Mention your goals (major, degree), need for scholarships, and extracurriculars.<br>…

Textarea(value='', description='Your Profile:', layout=Layout(height='120px', width='100%'), placeholder="Exam…

Button(button_style='primary', description='Get Recommendations', icon='💡', style=ButtonStyle(), tooltip='Run …

Output()

### 🧭 Final Thoughts & Next Steps
This project demonstrates how GenAI and Retrieval-Augmented Generation (RAG) can be combined to offer real-world impact in education access — especially for first-generation students navigating the complex college landscape.

By leveraging semantic search, structured outputs, and prompt engineering, we’ve created a personalized advisor that delivers:

🎯 Relevant college and scholarship matches

🧠 Psychologically aligned extracurricular ideas

📊 Transparent similarity scores to build trust in the recommendations

### ✅ Key Learnings
Grounding GenAI with real data dramatically improves relevance and accuracy

Simple UI tools like ipywidgets make powerful AI experiences accessible

FAISS + embeddings are incredibly effective for use cases like this where structured databases meet natural language

### 🔮 Future Directions
To take this further:

📥 Upload support: Let students upload transcripts or resumes for more context

🌐 Web version: Deploy a Streamlit or V0.dev app for public use

🧑‍🏫 Counselor mode: Add a guided version for school advisors or mentors

📈 Metrics: Track recommendation diversity and fairness across demographics

### 🙌 Thank You for Reading
If you found this useful or inspiring, feel free to:

⭐ Upvote the notebook

💬 Leave a comment or question

🚀 Fork it and build your own AI advisor

Let’s use AI not just to automate — but to amplify access and opportunity. 💙

